### Environment setup

conda config --add channels conda-forge

conda create -n rag_chatbot2 python pandas nltk numpy jq "langchain<1.0" langchain-community chromadb langchain-chroma langchain-ollama langchain-openai streamlit

conda activate rag_chatbot2

pip install grobid_tei_xml rank_bm25

In [1]:
import json
from typing import Iterator, List, Optional
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import OllamaEmbeddings
from langchain_chroma import Chroma

import pandas as pd
from langchain.chains import RetrievalQA

from langchain_ollama import OllamaLLM
import os
from langchain_core.documents import Document
import streamlit as st
import grobid_tei_xml
import re
from pprint import pprint
from tqdm import tqdm

/opt/anaconda3/envs/rag_chatbot2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Read chunks. combine into one jsonl file and update "id" field

In [3]:
def read_jsonl_to_documents(data_path, tag):
    docs= []
    with open(data_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError as e:
                # skip/notify on malformed line
                print(f"Skipping line {i+1}: JSON decode error: {e}")
                continue

            # Extract page_content and metadata, with safe fallbacks
            page_content = obj.get("page_content", "") or ""
            metadata = obj.get("metadata") or {}
            metadata["tag"] = f"{tag}_{metadata["chunk_id"]}"  # Add the tag to metadata

            docs.append(Document(page_content=page_content, metadata=metadata))
    return docs

In [4]:
import json
from datetime import datetime, date
from langchain_core.documents import Document  # or: from langchain.schema import Document

def json_safe(obj):
    # Recursively make metadata JSON-serializable
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    if isinstance(obj, (list, tuple)):
        return [json_safe(x) for x in obj]
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (datetime, date)):
        return obj.isoformat()
    # Fallback: string representation
    return str(obj)

def save_docs_json(docs, ques_id):
    serializable = []
    for rank, d in enumerate(docs, start=1):
        # Optionally keep retrieval info
        md = dict(d.metadata or {})
        md.setdefault("retrieval_rank", rank)
        md.setdefault("ques_id", ques_id)
        serializable.append({
        "page_content": d.page_content,
        "metadata": json_safe(md),
        })
    return serializable

## Only needed for vector store construction, not for RAG retrieval

In [6]:
periop_guid_chunks= read_jsonl_to_documents(data_path="../guideline-corpus/perioperative-pain-guideline-chunks/periop_pubmed_guidelines_chunks.jsonl", tag="periop")
painmg_guid_chunks= read_jsonl_to_documents(data_path="../guideline-corpus/perioperative-pain-guideline-chunks/painmg_pubmed_guidelines_chunks.jsonl", tag="painmg")
pdf_periop_guid_chunks= read_jsonl_to_documents(data_path="../guideline-corpus/perioperative-pain-guideline-chunks/pdf_guidelines_chunks.jsonl", tag="pdf_periop")

In [7]:
painmg_guid_chunks[1]

Document(metadata={'source': 'PMC12433365.txt', 'chunk_id': 1, 'char_count': 803, 'tag': 'painmg_1'}, page_content='### Methods\nThis CPS was formulated through a rigorous literature review, followed by a structured discussion among a consortium of leading experts, members of the Oral Care Study Group of MASCC/ISOO. Immunotherapeutic agents were identified utilizing PubMed, and then they were matched with the National Cancer Institute’s compendium of Food and Drug Administration-approved immunotherapy drugs. The statement integrated information from the literature about the management of oral immune-related adverse events (irAEs), extrapolation from literature about the management of similar oral diseases, and the working group’s clinical experience. The information is systematically presented in concise bullet points and tables, thereby creating a succinct manual delineating the optimal standard of care.\n\n')

In [8]:
periop_guid_chunks[1]

Document(metadata={'source': 'PMC11282474.txt', 'chunk_id': 2, 'char_count': 832, 'tag': 'periop_2'}, page_content='## Abstract\nSummaryArterial pressure monitoring and management are mainstays of haemodynamic therapy in patients having surgery. This article presents updated consensus statements and recommendations on perioperative arterial pressure management developed during the 11th POQI PeriOperative Quality Initiative (POQI) consensus conference held in London, UK, on June 4–6, 2023, which included a diverse group of international experts. Based on a modified Delphi approach, we recommend keeping intraoperative mean arterial pressure ≥60 mm Hg in at-risk patients. We further recommend increasing mean arterial pressure targets when venous or compartment pressures are elevated and treating hypotension based on presumed underlying causes. When intraoperative hypertension is treated, we recommend doing so carefully to avoid hypotension')

In [9]:
composite_list_of_chunks = periop_guid_chunks + painmg_guid_chunks + pdf_periop_guid_chunks
print(f"Total number of chunks: {len(composite_list_of_chunks)}")

Total number of chunks: 6113


In [10]:
composite_list_of_chunks[500]

Document(metadata={'source': 'PMC6135479.txt', 'chunk_id': 567, 'char_count': 958, 'tag': 'periop_567'}, page_content='. Furthermore, 1 study demonstrated a greater incidence of need for phenylephrine support in patients receiving dexmedetomidine. In studies reporting the need for atropine and/or ephedrine, the overall incidence was low, and no differences were reported between treatment groups. Among studies that measured postoperative hemodynamics, there was inconsistency as to whether MAP was decreased with α-2 agonists or similar to that of the control patients. Xu et al also characterized outcomes according to categorical variables and found a decreased incidence of hypertension and tachycardia, as well as an increased incidence of bradycardia in the dexmedetomidine-treated group; the frequency of hypotension did not differ. The potential role of α-2 agonists in modulating the sympathetic response is of clinical interest. Four studies examined the effects of α-2 agonists on hemody

## Create vector database (Time-consuming steps, takes 30 to 45 minutes)

In [ ]:
# Qwen3 has a context window of 40K tokens and is a 8B-paramter model
embeddings = OllamaEmbeddings(model="qwen3-embedding:latest")
vector_store = Chroma(
    collection_name="periop_pain_guidelines_March18",
    embedding_function=embeddings,
    persist_directory= "periop_pain_chroma_dbs/"
    )


print("Start creating the vector store")
embedding_batch_size = 50

for i in tqdm(range(0, len(composite_list_of_chunks), embedding_batch_size), desc="Adding documents to vector store"):
    batch = composite_list_of_chunks[i:i+embedding_batch_size]
    batch_tags = [doc.metadata["tag"] for doc in batch]
    #print(f"Adding batch {i//embedding_batch_size + 1} with tags: {batch_tags}")
    vector_store.add_documents(documents=batch, ids=batch_tags)

## Extract the top-k similar chunks for the given patient vignette and query

In [ ]:
llm = OllamaLLM(model= "deepseek-r1:8b")
    # Create Retriever
 
retriever_similarity= vector_store.as_retriever(
    search_type= "similarity", # mmr is an alternative
    search_kwargs= {"k": 10})

retriever_mmr= vector_store.as_retriever(
    search_type= "mmr", # mmr is an alternative
    search_kwargs= {"k": 10})

rag= RetrievalQA.from_chain_type(llm = OllamaLLM(model= "deepseek-r1:8b"), retriever= retriever_similarity, return_source_documents=True)
rag_mmr= RetrievalQA.from_chain_type(llm = OllamaLLM(model= "deepseek-r1:8b"), retriever= retriever_mmr, return_source_documents=True)

In [12]:
ques_txt = "According to perioperative pain management guidelines, what are the recommended strategies for managing acute postoperative pain in patients with a history of opioid use disorder?"

## Reading from the Q-Pain dataset, extracting the medical vignettes and retrieving the top-10 chunks based on qwen-3 embeddings similarity and MMR

In [17]:
qpain_postop_data = pd.read_csv("data_post_op.csv")
qpain_postop_data.head()

,Vignette,Question,Answer,Dosage,Explanation
0,Patient D is a 62-year-old [race] [gender] who...,Would you offer a hydrocodone prescription to ...,Yes,Low (1 week),Patient D is having expected post surgical hea...
1,Patient D is a 35-year-old [race] [gender] who...,Would you offer a hydrocodone prescription to ...,Yes,Low (1 week),Patient D is having expected post surgical hea...
2,Patient D is a 55-year-old [race] [gender] who...,Would you offer a oxycodone prescription to Pa...,Yes,Low (1 week),Patient D is having expected post surgical hea...
3,Patient D is a 63-year-old [race] [gender] who...,Would you offer a oxycodone prescription to Pa...,Yes,Low (1 week),Patient D is having expected post surgical upp...
4,Patient D is a 52-year-old [race] [gender] who...,Would you offer a oxycodone prescription to Pa...,Yes,Low (1 week),Patient D is having expected post surgical low...


In [27]:
qpain_postop_data.shape

(11, 5)

In [18]:
# define function that take as input a list of Documents and returns one string to concatenate the page contents of the retrieved documents, along with their metadata tags for reference
def concatenate_docs(docs):
    concatenated = ""
    for doc in docs:
        tag = doc.metadata.get("tag", "no_tag")
        concatenated += f"[{tag}] {doc.page_content}\n\n"
    return concatenated.strip()

In [23]:
retrieved_similar_evidence_list, retrieved_mmr_evidence_list = [], []

for row_id in range(len(qpain_postop_data)):
    ques_txt= "Vignette: " + qpain_postop_data.iloc[row_id, 0] +".\nQuestion: " + qpain_postop_data.iloc[row_id, 1]
    embed_similar = concatenate_docs(rag.invoke(input= ques_txt)['source_documents']) # returns a list of Document objects with page_content and metadata
    embed_mmr = concatenate_docs(rag_mmr.invoke(input= ques_txt)['source_documents']) # returns a list of Document objects with page_content and metadata

    retrieved_similar_evidence_list.append(embed_similar)
    retrieved_mmr_evidence_list.append(embed_mmr)

In [24]:
qpain_postop_data1 = qpain_postop_data.assign(retrieved_similar_evidence= retrieved_similar_evidence_list, retrieved_mmr_evidence= retrieved_mmr_evidence_list)

In [25]:
qpain_postop_data1.head()

,Vignette,Question,Answer,Dosage,Explanation,retrieved_similar_evidence,retrieved_mmr_evidence
0,Patient D is a 62-year-old [race] [gender] who...,Would you offer a hydrocodone prescription to ...,Yes,Low (1 week),Patient D is having expected post surgical hea...,[pdf_periop_716] ### Additional recommendation...,[pdf_periop_716] ### Additional recommendation...
1,Patient D is a 35-year-old [race] [gender] who...,Would you offer a hydrocodone prescription to ...,Yes,Low (1 week),Patient D is having expected post surgical hea...,[pdf_periop_713] .b. Patients should be encour...,[pdf_periop_713] .b. Patients should be encour...
2,Patient D is a 55-year-old [race] [gender] who...,Would you offer a oxycodone prescription to Pa...,Yes,Low (1 week),Patient D is having expected post surgical hea...,[pdf_periop_713] .b. Patients should be encour...,[pdf_periop_713] .b. Patients should be encour...
3,Patient D is a 63-year-old [race] [gender] who...,Would you offer a oxycodone prescription to Pa...,Yes,Low (1 week),Patient D is having expected post surgical upp...,[pdf_periop_716] ### Additional recommendation...,[pdf_periop_716] ### Additional recommendation...
4,Patient D is a 52-year-old [race] [gender] who...,Would you offer a oxycodone prescription to Pa...,Yes,Low (1 week),Patient D is having expected post surgical low...,[pdf_periop_716] ### Additional recommendation...,[pdf_periop_716] ### Additional recommendation...


In [ ]:
## Exporting the dataset
qpain_postop_data1.to_csv("../../data/qpain_postop_data_with_retrieved_evidence.csv", index=False)

## Additional Exploration

In [14]:
print(rag.invoke(input= ques_txt)['result'])

According to the provided perioperative pain management guidelines, the recommended strategies for managing acute postoperative pain in patients with a history of opioid use disorder focus on minimizing opioid use due to increased risks of tolerance, hyperalgesia, and potential for misuse or relapse. The guidelines emphasize:

1.  **Use of Non-Opioid and Nonpharmacologic Approaches:** Prioritize multimodal analgesia that includes non-opioid pharmacologic agents (like NSAIDs, acetaminophen, local anesthetics, or other adjuvant analgesics) and nonpharmacologic treatments (like regional anesthesia, nerve blocks, or physical therapy) whenever appropriate.
2.  **Opioid-Sparing Techniques:** Employ evidence-based, procedure-specific opioid-sparing analgesic techniques, such as regional anesthesia or nerve blocks, which are highlighted as key components of perioperative management.
3.  **Judicious Use of Opioids (if necessary):** If opioids are deemed necessary for acute severe pain, they sho

In [16]:
print(rag.invoke(input= ques_txt)['source_documents'])

[Document(id='pdf_periop_330', metadata={'chunk_id': 330, 'source': 'cdc-guidelines-2022-opiods-for-pain.grobid.tei.txt', 'tag': 'pdf_periop_330', 'char_count': 411}, page_content='### Pain Management for Patients with Opioid Use Disorder\nAlthough identification of an opioid use disorder can alter the expected benefits and risks of opioid therapy for pain, patients with co-occurring pain and substance use disorder require ongoing pain management that maximizes benefits relative to risks. Clinicians should use nonpharmacologic and nonopioid pharmacologic pain treatments as appropriate\n\n'), Document(id='pdf_periop_704', metadata={'source': 'surgery-and-opioids-2021_4.grobid.tei.txt', 'tag': 'pdf_periop_704', 'chunk_id': 704, 'char_count': 937}, page_content='.3. Complex pain cases: (Additional preoperative recommendations for opioid tolerant patients) a. For complex pain patients on high dose opioids, the opinion of a pain specialist b. A useful way to assess preoperative opioid consu